# MPI

MPI (Message Passing Interface) = A standard for parallel programming where multiple independent processes communicate by sending and receiving messages. Each process has its own private memory. No shared memory.


MPI runs at the process level.




# Some Conceptual Similarities with C Processes

| C Process Concept | MPI Equivalent | What It Does |
|------------------|----------------|---------------|
| `fork()` | `mpirun -np N (process launcher concept)` | Launch multiple processes |
| `pid` | `rank` | Unique process identifier |
| `getpid()` | `MPI_Comm_rank()` | Get this process's ID |
| `pipe()` | `MPI_Send()` / `MPI_Recv()` | Send/receive messages |
| `wait()` | `MPI_Barrier()` | Synchronize all processes |
| `exit()` | `MPI_Finalize()` | Clean shutdown |


# SPMD (Single Program, Multiple Data)


SPMD is a parallel programming model where:

- all processes execute the same program
- each process works on different data
- behavior changes based on process rank

---

## Basic Idea

One program → many processes

Each process:

- has its own rank
- performs its own portion of work
- runs simultaneously with others

---

## Simple Visualization
```bash
          SAME PROGRAM

    +-------------------+
    |    MPI Program    |
    +-------------------+

      /      |      \
     /       |       \

 Rank 0   Rank 1   Rank 2
 
 Data A   Data B   Data C
```
---

## Key Characteristics

- Single executable
- Multiple processes
- Distributed computation
- Parallel execution
- Rank-based behavior

---

## Important Terms

| Term | Meaning |
|------|----------|
| Rank | Unique process ID |
| Size | Total number of processes |
| Communicator | Group of MPI processes |
| Parallelism | Multiple tasks running together |

---

## Workflow

1. Start MPI
2. Get rank and size
3. Divide data
4. Perform local computation
5. Exchange information if needed
6. Combine results
7. Finalize MPI

---

## Why SPMD?

- Easy to scale
- Efficient for large computations
- Natural fit for distributed systems
- Foundation of MPI programming

---

## Real Applications

- Matrix multiplication
- Weather simulations
- Fluid dynamics
- AI training
- Scientific computing

---

## Core Concept

> Same code, different responsibilities.

Processes decide what to do using their rank.



# Code Example 

In [9]:
%%bash
cat << 'EOF' > code1.c
#include <mpi.h>
#include <stdio.h>

int main(int argc, char *argv[])
{
    int rank, size; //rank = this process ID, size = total processes

    MPI_Init(&argc, &argv); //Start MPI. Every MPI program must call this first.


    MPI_Comm_rank(MPI_COMM_WORLD, &rank); // "Who am I?" → rank = 0, 1, 2, ...
    MPI_Comm_size(MPI_COMM_WORLD, &size); // "How many of us?" → size = total processes

    printf("Hello from process %d out of %d\n", rank, size);

    MPI_Finalize(); // Clean up. Every MPI program must call this last.

    return 0;
}
EOF

/usr/bin/mpicc code1.c -o code1
/usr/bin/mpirun -np 2 ./code1

Hello from process 0 out of 2
Hello from process 1 out of 2


# MPI_Send and MPI_Recv (Point-to-Point Communication)

## What are they?

In MPI, **MPI_Send** and **MPI_Recv** are used for direct communication between two processes.

- One process sends data
- Another process receives data

This is called **point-to-point communication**

---

## Basic Idea

Process A → sends data → Process B receives data

Example:

- Rank 0 sends message
- Rank 1 receives message

---

## MPI_Send

### Purpose
Sends data from one process to another.

### Concept

- You specify:
  - data to send
  - destination rank
  - message tag

---

## MPI_Recv

### Purpose
Receives data from another process.

### Concept

- You specify:
  - buffer to store data
  - source rank
  - message tag

---

## Simple Communication Flow

```

Rank 0                Rank 1
|                     |
|---- MPI_Send -----> |
|                     |
| <--- MPI_Recv ----- |

```

---

## Key Parameters

### MPI_Send

- buffer → data to send
- count → number of elements
- datatype → type of data (int, float, etc.)
- dest → destination rank
- tag → message identifier
- communicator → usually MPI_COMM_WORLD

---

### MPI_Recv

- buffer → where data is stored
- count → max number of elements to receive
- datatype → type of data
- source → sender rank
- tag → must match sender tag
- status → info about received message

---

## Important Rules

- Sender and receiver must match:
  - same datatype
  - compatible tag
  - correct ranks

- Mismatch can cause:
  - deadlock
  - hanging program

---

## Simple Example Scenario

### Rank 0:
- sends number = 42 to Rank 1

### Rank 1:
- receives number and prints it

---


# Code example 

In [10]:
%%bash
cat << 'EOF' > code1.c

#include <mpi.h>
#include <stdio.h>

int main(int argc, char *argv[])
{
    int rank;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    if (rank == 0)
    {
        int msg = 42;
        MPI_Send(&msg, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
        printf("Rank 0 sent: %d\n", msg);
    }
    else if (rank == 1)
    {
        int msg;
        MPI_Recv(&msg, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        printf("Rank 1 received: %d\n", msg);
    //&msg	address of variable
    //1 	count	Receive 1 element
    //MPI_INT	datatype	Element type is integer
    //0 	source	Receive from rank 0 only
    //0 	 tag	Match tag 0 (for filtering messages)
    //MPI_COMM_WORLD	communicator	All processes in this group
    //MPI_STATUS_IGNORE 	 status 	 Ignore message metadata (sender, tag, etc.)

    
    }

    MPI_Finalize();
    return 0;
}

EOF

/usr/bin/mpicc code1.c -o code1
/usr/bin/mpirun -np 2 ./code1

Rank 0 sent: 42
Rank 1 received: 42


# MPI_Bcast (Broadcast Communication)

`MPI_Bcast()` is a collective communication operation that allows **one process to send the same data to all other processes** in a communicator.

Instead of sending the same message multiple times using `MPI_Send()`, a single broadcast operation distributes the data efficiently.

---

## Basic Idea

One process acts as the **root** and shares data with everyone else.

```text id="g1v8pa"
             Root (Rank 0)
                    |
        -------------------------
        |           |           |
        ▼           ▼           ▼
      Rank 1      Rank 2      Rank 3
````

---

## Why Use MPI_Bcast?

* Simpler than multiple send operations
* Efficient communication
* Ensures all processes receive identical data
* Commonly used for distributing input parameters and configuration data

---

## How It Works

1. The root process initializes the data.
2. `MPI_Bcast()` is called by **all processes**.
3. MPI automatically distributes the data.
4. Every process receives the same value.

---





# Code Example

In [16]:
%%bash
cat << 'EOF' > code1.c

#include <mpi.h>
#include <stdio.h>

int main(int argc, char *argv[])
{
    int rank;
    int number;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    if (rank == 0)
    {
        number = 100;
        printf("Rank 0 broadcasting: %d\n", number);
    }

    MPI_Bcast(&number, 1, MPI_INT, 0, MPI_COMM_WORLD);

    printf("Rank %d received: %d\n", rank, number);

    MPI_Finalize();
    return 0;
}


EOF

/usr/bin/mpicc code1.c -o code1
/usr/bin/mpirun --oversubscribe -np 4 ./code1
#--oversubscribe enables MPI to run more processes than the number of CPU cores.

Rank 0 broadcasting: 100
Rank 0 received: 100
Rank 1 received: 100
Rank 3 received: 100
Rank 2 received: 100



# MPI_Reduce (Collective Reduction Operation)


`MPI_Reduce()` is a collective operation where **all processes contribute data**, and the results are **combined into a single value at a root process**.

---

## Basic Idea

Each process has a value → MPI combines them → one process gets the final result

```text id="r8p1qa"
Rank 0:  10
Rank 1:  20
Rank 2:  30
Rank 3:  40
           │
           ▼
     MPI_Reduce (SUM)
           │
           ▼
Rank 0:  100
````

---

## How It Works

* Every process provides input data
* MPI applies an operation (like sum, max, min)
* Result is stored in the **root process**

---

## Common Operations

| Operation | Meaning               |
| --------- | --------------------- |
| MPI_SUM   | Sum of all values     |
| MPI_MAX   | Maximum value         |
| MPI_MIN   | Minimum value         |
| MPI_PROD  | Product of all values |
| MPI_LAND  | Logical AND           |
| MPI_LOR   | Logical OR            |

---



# Example Code 

In [19]:
%%bash
cat << 'EOF' > code1.c

#include <mpi.h>
#include <stdio.h>

int main(int argc, char *argv[])
{
    int rank, size;
    int value, sum;

    MPI_Init(&argc, &argv);

    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    // each process sets its own value
    value = rank + 1;

    printf("Rank %d has value %d\n", rank, value);

    // reduce all values into sum at root (rank 0)
    MPI_Reduce(&value, &sum, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0)
    {
        printf("Final sum at root = %d\n", sum);
    }

    MPI_Finalize();
    return 0;
}

EOF

/usr/bin/mpicc code1.c -o code1
/usr/bin/mpirun  --oversubscribe -np 4 ./code1


Rank 3 has value 4
Rank 1 has value 2
Rank 2 has value 3
Rank 0 has value 1
Final sum at root = 10



# MPI_Barrier (Synchronization Point)

`MPI_Barrier()` is a **synchronization operation** where all processes must reach the barrier before any of them can continue.

---

## Basic Idea

All processes stop → wait for each other → then continue together

```text id="b1n7qp"
Rank 0  ----\
Rank 1  -----|---- MPI_Barrier -----> continue
Rank 2  -----|
Rank 3  ----/
````

---

## Key Concept

> No process moves forward until ALL processes reach the barrier.

---

## Why MPI_Barrier is Used

* Synchronize parallel tasks
* Ensure fairness in execution timing
* Measure performance (timing code sections)
* Coordinate stages of computation

---





In [25]:
%%bash
cat << 'EOF' > code1.c
#include <mpi.h>
#include <stdio.h>
#include <unistd.h>

int main(int argc, char *argv[])
{
    int rank;
    double start, now;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    start = MPI_Wtime();

    // simulate staggered arrival
    sleep(rank);

    now = MPI_Wtime();
    printf("Rank %d ARRIVED at barrier (t=%.2f s)\n", rank, now - start);

    MPI_Barrier(MPI_COMM_WORLD);

    now = MPI_Wtime();
    printf("Rank %d LEFT barrier (t=%.2f s)\n", rank, now - start);

    MPI_Finalize();
    return 0;
}

EOF

/usr/bin/mpicc code1.c -o code1
/usr/bin/mpirun  --oversubscribe -np 4 ./code1

Rank 0 ARRIVED at barrier (t=0.00 s)
Rank 1 ARRIVED at barrier (t=1.00 s)
Rank 2 ARRIVED at barrier (t=2.00 s)
Rank 3 ARRIVED at barrier (t=3.00 s)
Rank 1 LEFT barrier (t=3.00 s)
Rank 2 LEFT barrier (t=3.00 s)
Rank 0 LEFT barrier (t=3.00 s)
Rank 3 LEFT barrier (t=3.00 s)
